In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Load datasets
df = pd.read_csv('data/Billionaires Statistics Dataset.csv')       # 2023 snapshot
df_hist = pd.read_csv('data/billionaires.csv')                     # historical 1996-2014

print(f'2023 snapshot : {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Historical    : {df_hist.shape[0]} rows, {df_hist.shape[1]} columns')

2023 snapshot : 2640 rows, 35 columns
Historical    : 2614 rows, 22 columns


Where do billionaire live (chloropeth map)

In [ ]:
# All billionaires by country - Choropleth Map
import plotly.graph_objects as go
import plotly.express as px

# Get all billionaires
all_billionaires = df.copy()

# Group by country and aggregate
billionaires_by_country = all_billionaires.groupby('country').agg({
    'finalWorth': 'sum',
    'personName': 'count'
}).reset_index()

# Rename columns for clarity
billionaires_by_country.columns = ['Country', 'Total_Worth', 'Count']

# Convert finalWorth to numeric (remove any non-numeric characters if present)
billionaires_by_country['Total_Worth'] = pd.to_numeric(billionaires_by_country['Total_Worth'], errors='coerce')




All Billionaires Distribution by Country:
           Country  Total_Worth  Count
74   United States      4575100    754
16           China      1805500    523
31           India       628700    157
24          France       499500     35
26         Germany       462100    102
65     Switzerland       409900     78
73  United Kingdom       370700     82
58          Russia       351000     79
29       Hong Kong       321500     68
13          Canada       173900     42

Total countries represented: 78


In [8]:
# Create Choropleth Map with Emerald Green Color
# First, map country names to ISO-3 codes
import pycountry

country_iso = {}
for idx, country in enumerate(billionaires_by_country['Country']):
    try:
        country_iso[country] = pycountry.countries.search_fuzzy(country)[0].alpha_3
    except:
        # Handle special cases
        special_cases = {
            'United States': 'USA',
            'United Kingdom': 'GBR',
            'China': 'CHN',
            'Hong Kong': 'HKG',
            'Taiwan': 'TWN',
            'Russia': 'RUS',
            'South Korea': 'KOR'
        }
        country_iso[country] = special_cases.get(country, country)

billionaires_by_country['ISO_Code'] = billionaires_by_country['Country'].map(country_iso)

# Emerald green palette
emerald_colorscale = [
    [0, '#FFFFFF'],           # White for low values
    [0.3, '#A8E6CF'],         # Light emerald
    [0.6, '#56D8A0'],         # Medium emerald
    [1, '#1B5E4A']            # Dark emerald for highest values
]

fig = go.Figure(data=go.Choropleth(
    locations=billionaires_by_country['ISO_Code'],
    z=billionaires_by_country['Total_Worth'],
    text=billionaires_by_country['Country'],
    customdata=billionaires_by_country[['Count', 'Total_Worth']],
    hovertemplate='<b>%{text}</b><br>' +
                  'Number of Billionaires: %{customdata[0]:.0f}<br>' +
                  'Total Wealth (USD Billions): %{customdata[1]:,.0f}<br>' +
                  '<extra></extra>',
    colorscale=emerald_colorscale,
    colorbar=dict(
        title="Total Wealth<br>(USD Billions)",
        thickness=15,
        len=0.7
    )
))

fig.update_layout(
    title_text='<b>All Billionaires Distribution by Country</b><br><sub>Wealth Aggregated by Country (Emerald Green = Greater Wealth)</sub>',
    geo=dict(
        projection_type='natural earth',
        showland=True,
        landcolor='rgb(243, 243, 243)',
        coastlinecolor='rgb(204, 204, 204)',
        showocean=True,
        oceancolor='rgb(100, 180, 220)'
    ),
    height=700,
    font=dict(size=11)
)

fig.show()

## Are Billionaire born on same day?

In [13]:
# Visualization - Top 25 Most Common Billionaire Birthdays
import plotly.graph_objects as go

# Get top 25 birthdays
top_birthdays = birthday_counts.head(25)

# Emerald green color gradient
emerald_colors = ['#1B5E4A' if i == 0 else '#56D8A0' if i < 12 else '#A8E6CF' 
                  for i in range(len(top_birthdays))]

fig = go.Figure(data=[
    go.Bar(
        y=top_birthdays['month_day'],
        x=top_birthdays['Count'],
        orientation='h',
        marker=dict(
            color=top_birthdays['Count'],
            colorscale=[
                [0, '#A8E6CF'],        # Light emerald
                [0.5, '#56D8A0'],      # Medium emerald
                [1, '#1B5E4A']         # Dark emerald
            ],
            showscale=True,
            colorbar=dict(
                title="Number of<br>Billionaires",
                thickness=15,
                len=0.7
            )
        ),
        hovertemplate='<b>%{y}</b><br>' +
                      'Billionaires: %{x}<br>' +
                      '<extra></extra>',
        text=top_birthdays['Count'],
        textposition='outside'
    )
])

fig.update_layout(
    title_text='<b>Top 25 Most Common Billionaire Birthdays</b><br><sub>Emerald Green Intensity = More Billionaires Share This Birthday</sub>',
    xaxis_title='Number of Billionaires',
    yaxis_title='Birthday (Month Day)',
    hovermode='y unified',
    height=700,
    width=900,
    template='plotly_white',
    font=dict(size=11),
    showlegend=False
)

fig.show()


## Top Billionaires Leaderboard

In [18]:
# Top 10 Billionaires Profile Cards - Visual Leaderboard with Plotly
import plotly.graph_objects as go

# Prepare data for visual cards
df_top_10 = df.nlargest(10, 'finalWorth')[['rank', 'personName', 'finalWorth', 'source', 'country']].copy()
df_top_10['finalWorth'] = pd.to_numeric(df_top_10['finalWorth'], errors='coerce')
df_top_10['finalWorth'] = df_top_10['finalWorth'].round(0).astype(int)
df_top_10 = df_top_10.reset_index(drop=True)

# Create a more visual representation using scatter plot styled as cards
fig = go.Figure()

# Add bars showing wealth
fig.add_trace(go.Bar(
    y=df_top_10['personName'],
    x=df_top_10['finalWorth'],
    orientation='h',
    marker=dict(
        color=df_top_10['finalWorth'],
        colorscale=[
            [0, '#A8E6CF'],        # Light emerald
            [0.5, '#56D8A0'],      # Medium emerald
            [1, '#1B5E4A']         # Dark emerald
        ],
        showscale=True,
        colorbar=dict(
            title="Wealth<br>(USD B)",
            thickness=15,
            len=0.7
        ),
        line=dict(color='#1B5E4A', width=2)
    ),
    text=[f"<b>#{int(rank)}</b><br>${val}B<br>{src}" 
          for rank, val, src in zip(df_top_10['rank'], df_top_10['finalWorth'], df_top_10['source'])],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>' +
                  'Rank: #%{customdata[0]}<br>' +
                  'Wealth: $%{x}B<br>' +
                  'Source: %{customdata[1]}<br>' +
                  'Country: %{customdata[2]}<br>' +
                  '<extra></extra>',
    customdata=df_top_10[['rank', 'source', 'country']].values
))

fig.update_layout(
    title_text='<b>Top 10 Billionaires Profile Cards</b><br><sub>Ranked by Total Wealth - Emerald Green Scale</sub>',
    xaxis_title='Total Wealth (USD Billions)',
    yaxis_title='Billionaire Name',
    height=700,
    width=1100,
    hovermode='closest',
    template='plotly_white',
    font=dict(size=12, color='#1B5E4A', family='Arial'),
    paper_bgcolor='#F0F8F5',
    plot_bgcolor='#FFFFFF',
    margin=dict(l=200, r=100, t=150, b=60),
    yaxis=dict(tickfont=dict(size=11, color='#1B5E4A', family='Arial')),
    xaxis=dict(tickfont=dict(size=10, color='#1B5E4A'))
)

fig.update_yaxes(autorange="reversed")

fig.show()



## Age Analysis of Billionaires

In [23]:
# Age Analysis - Data Preparation
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Prepare age data
df_age = df[['personName', 'age', 'finalWorth', 'country', 'source']].copy()
df_age['finalWorth'] = pd.to_numeric(df_age['finalWorth'], errors='coerce')
df_age = df_age.dropna(subset=['age', 'finalWorth'])

# Convert age to numeric
df_age['age'] = pd.to_numeric(df_age['age'], errors='coerce')
df_age = df_age.dropna(subset=['age'])

print("\n" + "="*80)
print("BILLIONAIRE AGE ANALYSIS - STATISTICS")
print("="*80)
print(f"\nTotal billionaires with age data: {len(df_age)}")
print(f"\nAge Statistics:")
print(f"  Minimum Age: {df_age['age'].min():.0f} years")
print(f"  Maximum Age: {df_age['age'].max():.0f} years")
print(f"  Mean Age: {df_age['age'].mean():.1f} years")
print(f"  Median Age: {df_age['age'].median():.1f} years")
print(f"  Mode Age: {df_age['age'].mode().values[0]:.0f} years")
print(f"  Std Dev: {df_age['age'].std():.1f} years")
print(f"\nWealth vs Age:")
print(f"  Average Wealth: ${df_age['finalWorth'].mean():,.0f}B")
print(f"  Correlation (Age vs Wealth): {df_age['age'].corr(df_age['finalWorth']):.3f}")

# Age groups analysis
df_age['age_group'] = pd.cut(df_age['age'], 
                             bins=[0, 30, 40, 50, 60, 70, 80, 100],
                             labels=['<30', '30-40', '40-50', '50-60', '60-70', '70-80', '80+'])

age_group_stats = df_age.groupby('age_group', observed=True).agg({
    'personName': 'count',
    'finalWorth': ['sum', 'mean', 'median']
}).round(0)

print(f"\nAge Group Distribution:")
print("-" * 80)
age_group_stats.columns = ['Count', 'Total_Wealth', 'Avg_Wealth', 'Median_Wealth']
print(age_group_stats.to_string())



BILLIONAIRE AGE ANALYSIS - STATISTICS

Total billionaires with age data: 2575

Age Statistics:
  Minimum Age: 18 years
  Maximum Age: 101 years
  Mean Age: 65.1 years
  Median Age: 65.0 years
  Mode Age: 60 years
  Std Dev: 13.3 years

Wealth vs Age:
  Average Wealth: $4,679B
  Correlation (Age vs Wealth): 0.067

Age Group Distribution:
--------------------------------------------------------------------------------
           Count  Total_Wealth  Avg_Wealth  Median_Wealth
age_group                                                
<30           15         64000     4267.00        1700.00
30-40         70        315300     4504.00        2000.00
40-50        239        865200     3620.00        2000.00
50-60        679       2918400     4298.00        2200.00
60-70        660       2859600     4333.00        2400.00
70-80        573       2860000     4991.00        2600.00
80+          338       2165300     6406.00        3000.00


In [28]:
# Visualization 5: Violin Plot - Age Distribution by Age Group
fig_violin = go.Figure()

for i, age_group in enumerate(['<30', '30-40', '40-50', '50-60', '60-70', '70-80', '80+']):
    data = df_age[df_age['age_group'] == age_group]['finalWorth']
    
    if len(data) > 0:
        fig_violin.add_trace(go.Violin(
            x=[age_group] * len(data),
            y=data,
            name=age_group,
            box_visible=True,
            meanline_visible=True,
            fillcolor=['#A8E6CF', '#B5EBD8', '#56D8A0', '#4ACCA0', '#1B5E4A', '#0D3D2E', '#043D2C'][i],
            line_color='#1B5E4A',
            opacity=0.8,
            hovertemplate='Age Group: %{x}<br>Wealth: $%{y:.0f}B<extra></extra>'
        ))

fig_violin.update_layout(
    title_text='<b>Wealth Distribution by Billionaire Age Group</b><br><sub>Violin Plot with Box & Mean Line</sub>',
    xaxis_title='Age Group',
    yaxis_title='Total Wealth (USD Billions)',
    height=600,
    width=1100,
    template='plotly_white',
    font=dict(size=11, color='#1B5E4A', family='Arial'),
    paper_bgcolor='#F0F8F5',
    plot_bgcolor='#FFFFFF',
    showlegend=False,
    hovermode='closest'
)

fig_violin.show()

print("\n" + "="*80)
print("AGE GROUP WEALTH ANALYSIS")
print("="*80)
print(df_age_group_summary.to_string(index=False))
print("="*80)



AGE GROUP WEALTH ANALYSIS
Age_Group  Count  Avg_Wealth  Avg_Age
      <30     15     4266.67    25.47
    30-40     70     4504.29    37.33
    40-50    239     3620.08    46.33
    50-60    679     4298.09    56.06
    60-70    660     4332.73    65.58
    70-80    573     4991.27    75.16
      80+    338     6406.21    86.25


## GDP vs Billionaire Count Analysis
Validate if richer countries produce more billionaires

In [30]:
# Data Preparation: GDP vs Billionaire Count
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Get billionaires by country
billionaires_count = df.groupby('country').agg({
    'personName': 'count',
    'finalWorth': 'sum'
}).reset_index()
billionaires_count.columns = ['Country', 'Billionaire_Count', 'Total_Wealth']
billionaires_count['Total_Wealth'] = pd.to_numeric(billionaires_count['Total_Wealth'], errors='coerce')

# GDP data (2023 estimates in USD Billions)
# Source: IMF World Economic Outlook
gdp_data = {
    'United States': 27360, 'China': 17734, 'Japan': 4231, 'Germany': 4080,
    'India': 3736, 'United Kingdom': 3332, 'France': 2783, 'Italy': 2010,
    'Brazil': 1920, 'Canada': 2117, 'South Korea': 1642, 'Spain': 1390,
    'Australia': 1687, 'Mexico': 1313, 'Netherlands': 1257, 'Saudi Arabia': 1108,
    'Switzerland': 957, 'Turkey': 1163, 'Poland': 688, 'Belgium': 688,
    'Sweden': 615, 'Argentina': 632, 'Austria': 516, 'Norway': 598,
    'Nigeria': 477, 'Iran': 374, 'Israel': 529, 'Ireland': 530,
    'United Arab Emirates': 509, 'Singapore': 592, 'Hong Kong': 397,
    'Thailand': 504, 'Malaysia': 438, 'Indonesia': 1119, 'Philippines': 548,
    'Vietnam': 429, 'Pakistan': 376, 'Bangladesh': 416, 'Egypt': 437,
    'Russia': 1860, 'Ukraine': 184, 'Czech Republic': 296, 'Portugal': 267,
    'Greece': 219, 'Chile': 314, 'Colombia': 331, 'Peru': 245,
    'New Zealand': 249, 'Denmark': 405, 'Finland': 298, 'Romania': 301,
    'Taiwan': 764, 'Hong Kong': 397, 'Kazakhstan': 239, 'Kuwait': 186,
    'Qatar': 240, 'Bahrain': 43, 'Oman': 120, 'Malta': 19,
    'Luxembourg': 84, 'Cyprus': 32, 'Sri Lanka': 84, 'Cyprus': 32,
    'Jordan': 45, 'Lebanon': 22
}

# Merge GDP with billionaire data
gdp_df = pd.DataFrame(list(gdp_data.items()), columns=['Country', 'GDP'])
analysis_df = billionaires_count.merge(gdp_df, on='Country', how='inner')

print("\n" + "="*100)
print("GDP vs BILLIONAIRE COUNT ANALYSIS")
print("="*100)
print(f"\nCountries analyzed: {len(analysis_df)}")
print(f"\nCorrelation between GDP and Billionaire Count: {analysis_df['GDP'].corr(analysis_df['Billionaire_Count']):.3f}")
print(f"Correlation between GDP and Total Wealth: {analysis_df['GDP'].corr(analysis_df['Total_Wealth']):.3f}")

print("\n" + "-"*100)
print("Top 15 Countries by GDP and Billionaire Count:")
print("-"*100)
top_analysis = analysis_df.nlargest(15, 'GDP')[['Country', 'GDP', 'Billionaire_Count', 'Total_Wealth']]
print(top_analysis.to_string(index=False))
print("="*100)



GDP vs BILLIONAIRE COUNT ANALYSIS

Countries analyzed: 55

Correlation between GDP and Billionaire Count: 0.985
Correlation between GDP and Total Wealth: 0.968

----------------------------------------------------------------------------------------------------
Top 15 Countries by GDP and Billionaire Count:
----------------------------------------------------------------------------------------------------
       Country   GDP  Billionaire_Count  Total_Wealth
 United States 27360                754       4575100
         China 17734                523       1805500
         Japan  4231                 38        146800
       Germany  4080                102        462100
         India  3736                157        628700
United Kingdom  3332                 82        370700
        France  2783                 35        499500
        Canada  2117                 42        173900
         Italy  2010                 55        156900
        Brazil  1920                 44        10

In [33]:
# Visualization 3: Bubble Chart - GDP per Billionaire vs Billionaire Efficiency
analysis_df['GDP_per_Billionaire'] = analysis_df['GDP'] / analysis_df['Billionaire_Count']

# Sort by GDP per billionaire to find countries that produce billionaires efficiently
efficiency_ranking = analysis_df[['Country', 'GDP', 'Billionaire_Count', 'GDP_per_Billionaire', 'Total_Wealth']].sort_values('GDP_per_Billionaire')

fig_bubble = go.Figure()

fig_bubble.add_trace(go.Scatter(
    x=analysis_df['GDP'],
    y=analysis_df['Billionaire_Count'],
    mode='markers',
    text=analysis_df['Country'],
    marker=dict(
        size=analysis_df['Billionaire_Count'] * 0.8,  # Size by billionaire count
        color=analysis_df['GDP_per_Billionaire'],
        colorscale=[
            [0, '#1B5E4A'],        # Dark emerald = efficient (low GDP per billionaire)
            [1, '#A8E6CF']         # Light emerald = less efficient (high GDP per billionaire)
        ],
        colorbar=dict(
            title="GDP per<br>Billionaire<br>(Billions USD)",
            thickness=15,
            len=0.7
        ),
        line=dict(color='#1B5E4A', width=1),
        showscale=True
    ),
    hovertemplate='<b>%{text}</b><br>' +
                  'GDP: $%{x:.0f}B<br>' +
                  'Billionaires: %{y}<br>' +
                  'GDP per Billionaire: $%{customdata[0]:.1f}B<br>' +
                  'Total Wealth: $%{customdata[1]:.0f}B<br>' +
                  '<extra></extra>',
    customdata=analysis_df[['GDP_per_Billionaire', 'Total_Wealth']]
))

fig_bubble.update_layout(
    title_text='<b>Billionaire Production Efficiency</b><br><sub>Dark Emerald = More Efficient at Creating Billionaires | Bubble Size = Billionaire Count</sub>',
    xaxis_title='Country GDP (USD Billions)',
    yaxis_title='Number of Billionaires',
    height=700,
    width=1200,
    template='plotly_white',
    font=dict(size=11, color='#1B5E4A'),
    paper_bgcolor='#F0F8F5',
    plot_bgcolor='#FFFFFF',
    hovermode='closest'
)

fig_bubble.show()

print("\n" + "="*100)
print("BILLIONAIRE PRODUCTION EFFICIENCY ANALYSIS")
print("="*100)
print("\nMost Efficient Countries (Lowest GDP per Billionaire):")
print("-"*100)
most_efficient = efficiency_ranking.head(10)
print(most_efficient.to_string(index=False))

print("\n\nLeast Efficient Countries (Highest GDP per Billionaire):")
print("-"*100)
least_efficient = efficiency_ranking.tail(10)
print(least_efficient.to_string(index=False))
print("="*100)

print("\n📊 KEY INSIGHTS:")
print("  • Countries with LOWER 'GDP per Billionaire' are MORE efficient at creating billionaires")
print("  • This shows which economies convert wealth into billionaire entrepreneurship most effectively")



BILLIONAIRE PRODUCTION EFFICIENCY ANALYSIS

Most Efficient Countries (Lowest GDP per Billionaire):
----------------------------------------------------------------------------------------------------
    Country  GDP  Billionaire_Count  GDP_per_Billionaire  Total_Wealth
  Hong Kong  397                 68                 5.84        321500
     Cyprus   32                  5                 6.40          9600
    Lebanon   22                  2                11.00          5600
Switzerland  957                 78                12.27        409900
  Singapore  592                 46                12.87        138000
     Taiwan  764                 43                17.77        117900
   Thailand  504                 28                18.00        100700
     Israel  529                 26                20.35         72500
     Russia 1860                 79                23.54        351000
     Sweden  615                 26                23.65         90900


Least Efficient 

## Education vs Billionaires Analysis
Does higher education enrollment correlate with billionaire creation?

In [2]:
# Data Preparation: Education vs Billionaire Count - COMPREHENSIVE ANALYSIS
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# Get billionaires and education data by country
df_education = df[['country', 'personName', 'finalWorth', 'gross_tertiary_education_enrollment', 'age', 'gender', 'source']].copy()
df_education['finalWorth'] = pd.to_numeric(df_education['finalWorth'], errors='coerce')
df_education['gross_tertiary_education_enrollment'] = pd.to_numeric(df_education['gross_tertiary_education_enrollment'], errors='coerce')

# Group by country
education_analysis = df_education.groupby('country').agg({
    'personName': 'count',
    'finalWorth': 'sum',
    'gross_tertiary_education_enrollment': 'first'
}).reset_index()

education_analysis.columns = ['Country', 'Billionaire_Count', 'Total_Wealth', 'Education_Enrollment']
education_analysis = education_analysis.dropna(subset=['Education_Enrollment'])

# Calculate statistics
correlation_count = education_analysis['Education_Enrollment'].corr(education_analysis['Billionaire_Count'])
correlation_wealth = education_analysis['Education_Enrollment'].corr(education_analysis['Total_Wealth'])

# Statistical significance test
slope, intercept, r_value, p_value, std_err = stats.linregress(education_analysis['Education_Enrollment'], 
                                                                education_analysis['Billionaire_Count'])

# Categorize countries by education level
education_analysis['Education_Level'] = pd.cut(education_analysis['Education_Enrollment'], 
                                               bins=[0, 30, 50, 70, 100],
                                               labels=['Low (<30%)', 'Medium (30-50%)', 'High (50-70%)', 'Very High (70%+)'])

print("\n" + "="*100)
print("🎓 EDUCATION vs BILLIONAIRE COUNT - COMPREHENSIVE ANALYSIS")
print("="*100)
print(f"\n📊 Dataset Overview:")
print(f"  • Countries analyzed: {len(education_analysis)}")
print(f"  • Total billionaires: {education_analysis['Billionaire_Count'].sum():.0f}")
print(f"  • Total wealth tracked: ${education_analysis['Total_Wealth'].sum():,.0f}B")

print(f"\n📈 Education Enrollment Statistics (% of Population):")
print(f"  • Minimum: {education_analysis['Education_Enrollment'].min():.1f}%")
print(f"  • Maximum: {education_analysis['Education_Enrollment'].max():.1f}%")
print(f"  • Mean: {education_analysis['Education_Enrollment'].mean():.1f}%")
print(f"  • Median: {education_analysis['Education_Enrollment'].median():.1f}%")
print(f"  • Std Dev: {education_analysis['Education_Enrollment'].std():.1f}%")

print(f"\n🔗 CORRELATION ANALYSIS:")
print(f"  • Education vs Billionaire Count: {correlation_count:.3f}")
print(f"  • Education vs Total Wealth: {correlation_wealth:.3f}")
print(f"  • R-squared (Count): {r_value**2:.3f}")
print(f"  • P-value (statistical significance): {p_value:.6f}")
if p_value < 0.05:
    print(f"  ✅ SIGNIFICANT correlation (p < 0.05)")
else:
    print(f"  ⚠️  Weak statistical significance (p >= 0.05)")

print(f"\n💡 INTERPRETATION:")
if correlation_count > 0.5:
    print(f"  Strong POSITIVE correlation: Higher education → More billionaires")
elif correlation_count > 0.3:
    print(f"  Moderate POSITIVE correlation: Some relationship exists")
elif correlation_count > 0:
    print(f"  Weak POSITIVE correlation: Minimal relationship")
else:
    print(f"  NEGATIVE or NO correlation: Education may not drive billionaire creation")

print("\n" + "-"*100)
print("📊 BILLIONAIRE DISTRIBUTION BY EDUCATION LEVEL:")
print("-"*100)
edu_dist = education_analysis.groupby('Education_Level', observed=True).agg({
    'Country': 'count',
    'Billionaire_Count': 'sum',
    'Total_Wealth': 'sum',
    'Education_Enrollment': 'mean'
}).round(1)
edu_dist.columns = ['Countries', 'Billionaires', 'Total Wealth (B)', 'Avg Education %']
print(edu_dist)

print("\n" + "-"*100)
print("🏆 TOP 15 COUNTRIES BY EDUCATION ENROLLMENT:")
print("-"*100)
top_education = education_analysis.nlargest(15, 'Education_Enrollment')[['Country', 'Education_Enrollment', 'Billionaire_Count', 'Total_Wealth']]
for idx, row in top_education.iterrows():
    print(f"{row['Country']:20} | Enrollment: {row['Education_Enrollment']:6.1f}% | Billionaires: {row['Billionaire_Count']:3.0f} | Wealth: ${row['Total_Wealth']:8.0f}B")

print("\n" + "-"*100)
print("🤔 INTERESTING OUTLIERS:")
print("-"*100)
# Find outliers: low education but many billionaires
education_analysis['Billionaires_per_Enrollment'] = education_analysis['Billionaire_Count'] / (education_analysis['Education_Enrollment'] + 0.1)
outliers_high = education_analysis.nlargest(5, 'Billionaires_per_Enrollment')[['Country', 'Education_Enrollment', 'Billionaire_Count']]
print("Countries with HIGH billionaire production despite LOWER education:")
for idx, row in outliers_high.iterrows():
    print(f"  • {row['Country']:20} | {row['Education_Enrollment']:5.1f}% education | {row['Billionaire_Count']:3.0f} billionaires")

print("\n" + "="*100)



🎓 EDUCATION vs BILLIONAIRE COUNT - COMPREHENSIVE ANALYSIS

📊 Dataset Overview:
  • Countries analyzed: 66
  • Total billionaires: 2458
  • Total wealth tracked: $11,558,500B

📈 Education Enrollment Statistics (% of Population):
  • Minimum: 4.0%
  • Maximum: 136.6%
  • Mean: 57.5%
  • Median: 60.9%
  • Std Dev: 26.9%

🔗 CORRELATION ANALYSIS:
  • Education vs Billionaire Count: 0.122
  • Education vs Total Wealth: 0.140
  • R-squared (Count): 0.015
  • P-value (statistical significance): 0.330741
  ⚠️  Weak statistical significance (p >= 0.05)

💡 INTERPRETATION:
  Weak POSITIVE correlation: Minimal relationship

----------------------------------------------------------------------------------------------------
📊 BILLIONAIRE DISTRIBUTION BY EDUCATION LEVEL:
----------------------------------------------------------------------------------------------------
                  Countries  Billionaires  Total Wealth (B)  Avg Education %
Education_Level                                       

### Visualization 4: Dual Axis - Average Billionaire Wealth vs Education Enrollment

In [8]:
# Visualization 4: Dual Axis - Average Wealth per Billionaire vs Education
edu_sorted = education_analysis.sort_values('Education_Enrollment')
edu_sorted['Avg_Wealth_per_Billionaire'] = edu_sorted['Total_Wealth'] / edu_sorted['Billionaire_Count']

fig4 = make_subplots(
    rows=1, cols=1,
    specs=[[{"secondary_y": True}]]
)

# Add bars for average wealth
fig4.add_trace(
    go.Bar(
        x=edu_sorted['Country'],
        y=edu_sorted['Avg_Wealth_per_Billionaire'],
        name='Avg Wealth per Billionaire',
        marker=dict(color='#A8E6CF', line=dict(color='#1B5E4A', width=1)),
        hovertemplate='<b>%{x}</b><br>Avg Wealth: $%{y:.1f}B<extra></extra>'
    ),
    secondary_y=False,
)

# Add line for education enrollment
fig4.add_trace(
    go.Scatter(
        x=edu_sorted['Country'],
        y=edu_sorted['Education_Enrollment'],
        name='Education Enrollment %',
        mode='lines+markers',
        line=dict(color='#1B5E4A', width=3),
        marker=dict(size=8, color='#1B5E4A', symbol='diamond'),
        hovertemplate='<b>%{x}</b><br>Education: %{y:.1f}%<extra></extra>'
    ),
    secondary_y=True,
)

fig4.update_xaxes(title_text="Countries (sorted by Education Enrollment)")
fig4.update_yaxes(title_text="<b style='color:#A8E6CF'>Average Wealth per Billionaire (USD B)</b>", secondary_y=False)
fig4.update_yaxes(title_text="<b style='color:#1B5E4A'>Education Enrollment (%)</b>", secondary_y=True)

fig4.update_layout(
    title_text='<b>Average Billionaire Wealth vs Education Enrollment by Country</b><br>' +
               '<sub>Can Higher Education Create Wealthier Billionaires?</sub>',
    height=700,
    width=1400,
    template='plotly_white',
    font=dict(size=10, color='#1B5E4A'),
    paper_bgcolor='#F0F8F5',
    plot_bgcolor='#FFFFFF',
    hovermode='x unified',
    legend=dict(x=0.5, y=1.15, orientation='h', xanchor='center', yanchor='top')
)

fig4.show()



## Tax Rate vs Billionaire Density Analysis
Does higher taxation discourage billionaire creation? Examining the relationship between top income tax rates and billionaire density per capita.


In [9]:
# Data Preparation: Tax Rate vs Billionaire Density Analysis
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# Get billionaires by country with population data
df_tax = df[['country', 'personName', 'finalWorth']].copy()
df_tax['finalWorth'] = pd.to_numeric(df_tax['finalWorth'], errors='coerce')

billionaires_by_country = df_tax.groupby('country').agg({
    'personName': 'count',
    'finalWorth': 'sum'
}).reset_index()
billionaires_by_country.columns = ['Country', 'Billionaire_Count', 'Total_Wealth']

# Top income tax rates by country (2023-2024, latest available)
# Source: KPMG, PWC, National Tax Authorities
tax_rates = {
    'United States': 37.0, 'China': 45.0, 'Japan': 45.0, 'Germany': 42.0,
    'India': 42.0, 'United Kingdom': 45.0, 'France': 45.0, 'Italy': 43.0,
    'Brazil': 27.5, 'Canada': 53.53, 'South Korea': 45.0, 'Spain': 45.0,
    'Australia': 45.0, 'Mexico': 35.0, 'Netherlands': 49.5, 'Saudi Arabia': 0.0,
    'Switzerland': 35.0, 'Turkey': 40.0, 'Poland': 32.0, 'Belgium': 45.0,
    'Sweden': 57.0, 'Argentina': 35.0, 'Austria': 55.0, 'Norway': 48.8,
    'Nigeria': 24.0, 'Iran': 45.0, 'Israel': 50.0, 'Ireland': 40.0,
    'United Arab Emirates': 0.0, 'Singapore': 22.0, 'Hong Kong': 17.0,
    'Thailand': 37.0, 'Malaysia': 32.0, 'Indonesia': 30.0, 'Philippines': 32.0,
    'Vietnam': 35.0, 'Pakistan': 35.0, 'Bangladesh': 25.0, 'Egypt': 22.5,
    'Russia': 13.0, 'Ukraine': 18.0, 'Czech Republic': 32.0, 'Portugal': 48.0,
    'Greece': 44.0, 'Chile': 37.0, 'Colombia': 39.0, 'Peru': 30.0,
    'New Zealand': 39.0, 'Denmark': 56.0, 'Finland': 49.75, 'Romania': 40.0,
    'Taiwan': 40.0, 'Kazakhstan': 10.0, 'Kuwait': 0.0, 'Qatar': 0.0,
    'Bahrain': 0.0, 'Oman': 0.0, 'Malta': 35.0, 'Cyprus': 0.0,
    'Sri Lanka': 28.0, 'Jordan': 30.0, 'Lebanon': 20.0, 'Luxembourg': 45.0,
    'Iceland': 46.24, 'Greece': 44.0, 'Croatia': 25.0, 'Slovenia': 50.0,
    'Slovakia': 32.0, 'Hungary': 15.0, 'Lithuania': 32.0, 'Bulgaria': 10.0,
    'Morocco': 38.0, 'Tunisia': 35.0, 'Kenya': 30.0, 'South Africa': 45.0
}

# Population data (2023 estimates in millions)
population_data = {
    'United States': 338, 'China': 1425, 'Japan': 125, 'Germany': 84,
    'India': 1417, 'United Kingdom': 68, 'France': 68, 'Italy': 57,
    'Brazil': 216, 'Canada': 39, 'South Korea': 52, 'Spain': 48,
    'Australia': 26, 'Mexico': 130, 'Netherlands': 18, 'Saudi Arabia': 36,
    'Switzerland': 8.7, 'Turkey': 86, 'Poland': 38, 'Belgium': 12,
    'Sweden': 10, 'Argentina': 46, 'Austria': 9, 'Norway': 5.5,
    'Nigeria': 223, 'Iran': 89, 'Israel': 9, 'Ireland': 5,
    'United Arab Emirates': 9.9, 'Singapore': 5.9, 'Hong Kong': 7.5,
    'Thailand': 71, 'Malaysia': 34, 'Indonesia': 275, 'Philippines': 120,
    'Vietnam': 99, 'Pakistan': 240, 'Bangladesh': 170, 'Egypt': 105,
    'Russia': 146, 'Ukraine': 38, 'Czech Republic': 10, 'Portugal': 10,
    'Greece': 10, 'Chile': 19, 'Colombia': 52, 'Peru': 34,
    'New Zealand': 5.1, 'Denmark': 5.9, 'Finland': 5.6, 'Romania': 19,
    'Taiwan': 23, 'Kazakhstan': 20, 'Kuwait': 4.3, 'Qatar': 3,
    'Bahrain': 1.8, 'Oman': 4.6, 'Malta': 0.54, 'Cyprus': 1.2,
    'Sri Lanka': 22, 'Jordan': 10, 'Lebanon': 5, 'Luxembourg': 0.67,
    'Iceland': 0.39, 'Croatia': 3.9, 'Slovenia': 2.1, 'Slovakia': 5.5,
    'Hungary': 9.7, 'Lithuania': 2.8, 'Bulgaria': 6.9, 'Morocco': 37,
    'Tunisia': 12, 'Kenya': 54, 'South Africa': 60
}

# Merge data
tax_df = pd.DataFrame(list(tax_rates.items()), columns=['Country', 'Top_Tax_Rate'])
pop_df = pd.DataFrame(list(population_data.items()), columns=['Country', 'Population_Millions'])

# Combine all data
tax_analysis = billionaires_by_country.merge(tax_df, on='Country', how='inner')
tax_analysis = tax_analysis.merge(pop_df, on='Country', how='inner')

# Calculate billionaire density (per million people)
tax_analysis['Billionaire_Density'] = tax_analysis['Billionaire_Count'] / tax_analysis['Population_Millions']

# Calculate statistics
correlation_tax = tax_analysis['Top_Tax_Rate'].corr(tax_analysis['Billionaire_Count'])
correlation_tax_density = tax_analysis['Top_Tax_Rate'].corr(tax_analysis['Billionaire_Density'])

# Perform regression analysis
slope_tax, intercept_tax, r_value_tax, p_value_tax, std_err_tax = stats.linregress(
    tax_analysis['Top_Tax_Rate'], 
    tax_analysis['Billionaire_Density']
)

# Categorize countries by tax rate
tax_analysis['Tax_Level'] = pd.cut(tax_analysis['Top_Tax_Rate'], 
                                   bins=[-1, 15, 30, 45, 60],
                                   labels=['Very Low (0-15%)', 'Low (15-30%)', 'Medium (30-45%)', 'High (45%+)'])

print("\n" + "="*120)
print("💰 TAX RATE vs BILLIONAIRE DENSITY - COMPREHENSIVE ANALYSIS")
print("="*120)
print(f"\n📊 Dataset Overview:")
print(f"  • Countries analyzed: {len(tax_analysis)}")
print(f"  • Total billionaires: {tax_analysis['Billionaire_Count'].sum():.0f}")
print(f"  • Total population covered: {tax_analysis['Population_Millions'].sum():.0f}M people")

print(f"\n📈 Tax Rate Statistics (% of Income):")
print(f"  • Minimum: {tax_analysis['Top_Tax_Rate'].min():.1f}%")
print(f"  • Maximum: {tax_analysis['Top_Tax_Rate'].max():.1f}%")
print(f"  • Mean: {tax_analysis['Top_Tax_Rate'].mean():.1f}%")
print(f"  • Median: {tax_analysis['Top_Tax_Rate'].median():.1f}%")
print(f"  • Std Dev: {tax_analysis['Top_Tax_Rate'].std():.1f}%")

print(f"\n🌍 Billionaire Density Statistics (per Million People):")
print(f"  • Minimum: {tax_analysis['Billionaire_Density'].min():.3f}")
print(f"  • Maximum: {tax_analysis['Billionaire_Density'].max():.3f}")
print(f"  • Mean: {tax_analysis['Billionaire_Density'].mean():.3f}")
print(f"  • Median: {tax_analysis['Billionaire_Density'].median():.3f}")

print(f"\n🔗 CORRELATION ANALYSIS:")
print(f"  • Tax Rate vs Billionaire COUNT: {correlation_tax:+.3f}")
print(f"  • Tax Rate vs Billionaire DENSITY: {correlation_tax_density:+.3f}")
print(f"  • R-squared (Density): {r_value_tax**2:.3f}")
print(f"  • P-value: {p_value_tax:.6f}")
if p_value_tax < 0.05:
    print(f"  ✅ SIGNIFICANT correlation (p < 0.05)")
else:
    print(f"  ⚠️  Not statistically significant (p >= 0.05)")

print(f"\n💡 INTERPRETATION:")
if correlation_tax_density < -0.5:
    print(f"  ❌ STRONG NEGATIVE: Higher taxes → FEWER billionaires per capita")
    print(f"     → Tax policy appears to significantly discourage billionaire creation")
elif correlation_tax_density < -0.3:
    print(f"  📉 MODERATE NEGATIVE: Some inverse relationship exists")
    print(f"     → Higher taxes moderately correlate with fewer billionaires")
elif correlation_tax_density < 0:
    print(f"  📊 WEAK NEGATIVE: Minimal negative relationship")
elif correlation_tax_density > 0.3:
    print(f"  📈 POSITIVE: Higher taxes → MORE billionaires (counterintuitive!)")
else:
    print(f"  ➡️  NO meaningful correlation: Tax rates don't predict billionaire density")

print(f"\n" + "-"*120)
print("📊 BILLIONAIRE DISTRIBUTION BY TAX LEVEL:")
print("-"*120)
tax_dist = tax_analysis.groupby('Tax_Level', observed=True).agg({
    'Country': 'count',
    'Billionaire_Count': 'sum',
    'Billionaire_Density': 'mean',
    'Top_Tax_Rate': 'mean'
}).round(3)
tax_dist.columns = ['Countries', 'Billionaires', 'Avg Density/M', 'Avg Tax %']
print(tax_dist)

print("\n" + "-"*120)
print("🏆 TOP 10 COUNTRIES BY BILLIONAIRE DENSITY:")
print("-"*120)
top_density = tax_analysis.nlargest(10, 'Billionaire_Density')[['Country', 'Top_Tax_Rate', 'Billionaire_Count', 'Population_Millions', 'Billionaire_Density']]
for idx, row in top_density.iterrows():
    print(f"{row['Country']:20} | Tax: {row['Top_Tax_Rate']:5.1f}% | Billionaires: {row['Billionaire_Count']:4.0f} | Pop: {row['Population_Millions']:6.1f}M | Density: {row['Billionaire_Density']:.4f}")

print("\n" + "-"*120)
print("🎯 TAX POLICY EXTREMES:")
print("-"*120)

# Zero tax countries
zero_tax = tax_analysis[tax_analysis['Top_Tax_Rate'] == 0].nlargest(3, 'Billionaire_Density')
if len(zero_tax) > 0:
    print("Zero-Tax Havens - Billionaire Production:")
    for idx, row in zero_tax.iterrows():
        print(f"  • {row['Country']:20} | {row['Billionaire_Count']:3.0f} billionaires | Density: {row['Billionaire_Density']:.4f} per M")

# High tax countries
high_tax = tax_analysis[tax_analysis['Top_Tax_Rate'] >= 50].nlargest(3, 'Billionaire_Density')
if len(high_tax) > 0:
    print("\nHigh-Tax Countries (50%+) - Billionaire Production:")
    for idx, row in high_tax.iterrows():
        print(f"  • {row['Country']:20} | {row['Billionaire_Count']:3.0f} billionaires | Density: {row['Billionaire_Density']:.4f} per M")

print("\n" + "="*120)



💰 TAX RATE vs BILLIONAIRE DENSITY - COMPREHENSIVE ANALYSIS

📊 Dataset Overview:
  • Countries analyzed: 59
  • Total billionaires: 2563
  • Total population covered: 5817M people

📈 Tax Rate Statistics (% of Income):
  • Minimum: 0.0%
  • Maximum: 57.0%
  • Mean: 34.3%
  • Median: 38.0%
  • Std Dev: 15.0%

🌍 Billionaire Density Statistics (per Million People):
  • Minimum: 0.013
  • Maximum: 9.067
  • Mean: 1.115
  • Median: 0.394

🔗 CORRELATION ANALYSIS:
  • Tax Rate vs Billionaire COUNT: +0.105
  • Tax Rate vs Billionaire DENSITY: -0.111
  • R-squared (Density): 0.012
  • P-value: 0.403751
  ⚠️  Not statistically significant (p >= 0.05)

💡 INTERPRETATION:
  📊 WEAK NEGATIVE: Minimal negative relationship

------------------------------------------------------------------------------------------------------------------------
📊 BILLIONAIRE DISTRIBUTION BY TAX LEVEL:
------------------------------------------------------------------------------------------------------------------------


### Visualization 2: Bubble Chart - Tax Rate, Population, and Billionaires

In [11]:
# Visualization 2: Bubble Chart - Tax vs Population vs Billionaires
fig_tax2 = go.Figure()

fig_tax2.add_trace(go.Scatter(
    x=tax_analysis['Top_Tax_Rate'],
    y=tax_analysis['Billionaire_Density'],
    mode='markers',
    text=tax_analysis['Country'],
    marker=dict(
        size=tax_analysis['Population_Millions'] ** 0.6,  # Size by population
        color=tax_analysis['Billionaire_Count'],
        colorscale=[
            [0, '#A8E6CF'],        # Light emerald
            [0.5, '#56D8A0'],      # Medium emerald
            [1, '#1B5E4A']         # Dark emerald
        ],
        showscale=True,
        colorbar=dict(
            title="Billionaire<br>Count",
            thickness=15,
            len=0.7
        ),
        line=dict(color='white', width=2),
        opacity=0.8,
        sizemode='diameter'
    ),
    hovertemplate='<b>%{text}</b><br>' +
                  'Tax Rate: %{x:.1f}%<br>' +
                  'Billionaire Density: %{y:.4f} per M<br>' +
                  'Population: %{customdata[0]:.1f}M<br>' +
                  'Total Billionaires: %{customdata[1]:.0f}<br>' +
                  '<extra></extra>',
    customdata=tax_analysis[['Population_Millions', 'Billionaire_Count']]
))

fig_tax2.update_layout(
    title_text='<b>Tax Rate vs Billionaire Density by Country</b><br>' +
               '<sub>Bubble Size = Population | Color Intensity = Billionaire Count</sub>',
    xaxis_title='Top Income Tax Rate (%)',
    yaxis_title='Billionaire Density (per Million Population)',
    height=700,
    width=1200,
    template='plotly_white',
    font=dict(size=11, color='#1B5E4A'),
    paper_bgcolor='#F0F8F5',
    plot_bgcolor='#FFFFFF',
    hovermode='closest'
)

fig_tax2.show()

print("\n🌍 POPULATION-ADJUSTED ANALYSIS:")
tax_top_density = tax_analysis.nlargest(5, 'Billionaire_Density')[['Country', 'Top_Tax_Rate', 'Population_Millions', 'Billionaire_Count', 'Billionaire_Density']]
print("Countries with HIGHEST billionaire density (per capita):")
for idx, row in tax_top_density.iterrows():
    print(f"  • {row['Country']:20} | Tax: {row['Top_Tax_Rate']:5.1f}% | Pop: {row['Population_Millions']:6.1f}M | Density: {row['Billionaire_Density']:.4f}")



🌍 POPULATION-ADJUSTED ANALYSIS:
Countries with HIGHEST billionaire density (per capita):
  • Hong Kong            | Tax:  17.0% | Pop:    7.5M | Density: 9.0667
  • Switzerland          | Tax:  35.0% | Pop:    8.7M | Density: 8.9655
  • Singapore            | Tax:  22.0% | Pop:    5.9M | Density: 7.7966
  • Cyprus               | Tax:   0.0% | Pop:    1.2M | Density: 4.1667
  • Israel               | Tax:  50.0% | Pop:    9.0M | Density: 2.8889


In [13]:
# Visualization 4: Dual Comparison - Tax Rates grouped by Billionaire Output
tax_sorted = tax_analysis.sort_values('Top_Tax_Rate')

fig_tax4 = make_subplots(
    rows=1, cols=1,
    specs=[[{"secondary_y": True}]]
)

# Add bars for billionaire count
fig_tax4.add_trace(
    go.Bar(
        x=tax_sorted['Country'],
        y=tax_sorted['Billionaire_Count'],
        name='Billionaire Count',
        marker=dict(color='#A8E6CF', line=dict(color='#1B5E4A', width=1)),
        hovertemplate='<b>%{x}</b><br>Billionaires: %{y:.0f}<extra></extra>'
    ),
    secondary_y=False,
)

# Add line for tax rate
fig_tax4.add_trace(
    go.Scatter(
        x=tax_sorted['Country'],
        y=tax_sorted['Top_Tax_Rate'],
        name='Tax Rate %',
        mode='lines+markers',
        line=dict(color='#1B5E4A', width=3),
        marker=dict(size=6, color='#1B5E4A', symbol='circle'),
        hovertemplate='<b>%{x}</b><br>Tax: %{y:.1f}%<extra></extra>'
    ),
    secondary_y=True,
)

fig_tax4.update_xaxes(title_text="Countries (sorted by Tax Rate)")
fig_tax4.update_yaxes(title_text="<b style='color:#A8E6CF'>Number of Billionaires</b>", secondary_y=False)
fig_tax4.update_yaxes(title_text="<b style='color:#1B5E4A'>Top Income Tax Rate (%)</b>", secondary_y=True)

fig_tax4.update_layout(
    title_text='<b>Tax Rates vs Billionaire Count by Country</b><br>' +
               '<sub>Do Countries with Higher Taxes Have Fewer Billionaires?</sub>',
    height=650,
    width=1400,
    template='plotly_white',
    font=dict(size=9, color='#1B5E4A'),
    paper_bgcolor='#F0F8F5',
    plot_bgcolor='#FFFFFF',
    hovermode='x unified',
    legend=dict(x=0.5, y=1.15, orientation='h', xanchor='center', yanchor='top'),
    margin=dict(b=120)
)

fig_tax4.show()

print("\n💼 COMPARATIVE INSIGHTS:")
print(f"  • Correlation between Tax Rate and Count: {correlation_tax:.3f}")
print(f"  • Countries with tax ≤ 15%: {len(tax_analysis[tax_analysis['Top_Tax_Rate'] <= 15])} countries, {tax_analysis[tax_analysis['Top_Tax_Rate'] <= 15]['Billionaire_Count'].sum():.0f} billionaires")
print(f"  • Countries with tax ≥ 45%: {len(tax_analysis[tax_analysis['Top_Tax_Rate'] >= 45])} countries, {tax_analysis[tax_analysis['Top_Tax_Rate'] >= 45]['Billionaire_Count'].sum():.0f} billionaires")



💼 COMPARATIVE INSIGHTS:
  • Correlation between Tax Rate and Count: 0.105
  • Countries with tax ≤ 15%: 8 countries, 115 billionaires
  • Countries with tax ≥ 45%: 19 countries, 923 billionaires


## Life Expectancy vs Wealth Concentration Analysis
Are billionaires concentrated in healthier countries? Examining the relationship between life expectancy and billionaire creation patterns.


In [ ]:
# Data Preparation: Life Expectancy vs Wealth Concentration Analysis
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# Get billionaires data with life expectancy
df_life = df[['country', 'personName', 'finalWorth', 'life_expectancy_country']].copy()
df_life['finalWorth'] = pd.to_numeric(df_life['finalWorth'], errors='coerce')
df_life['life_expectancy_country'] = pd.to_numeric(df_life['life_expectancy_country'], errors='coerce')

# Group by country
life_analysis = df_life.groupby('country').agg({
    'personName': 'count',
    'finalWorth': 'sum',
    'life_expectancy_country': 'first'
}).reset_index()

life_analysis.columns = ['Country', 'Billionaire_Count', 'Total_Wealth', 'Life_Expectancy']
life_analysis = life_analysis.dropna(subset=['Life_Expectancy'])

# Convert total wealth to numeric
life_analysis['Total_Wealth'] = pd.to_numeric(life_analysis['Total_Wealth'], errors='coerce')
life_analysis = life_analysis.dropna(subset=['Total_Wealth'])

# Calculate wealth metrics
life_analysis['Avg_Wealth_per_Billionaire'] = life_analysis['Total_Wealth'] / life_analysis['Billionaire_Count']
life_analysis['Wealth_per_Million'] = life_analysis['Total_Wealth'] / 1000  # Wealth concentration metric

# Calculate correlations
corr_life_count = life_analysis['Life_Expectancy'].corr(life_analysis['Billionaire_Count'])
corr_life_wealth = life_analysis['Life_Expectancy'].corr(life_analysis['Total_Wealth'])
corr_life_avg = life_analysis['Life_Expectancy'].corr(life_analysis['Avg_Wealth_per_Billionaire'])

# Regression analysis
slope_life, intercept_life, r_value_life, p_value_life, std_err_life = stats.linregress(
    life_analysis['Life_Expectancy'], 
    life_analysis['Billionaire_Count']
)

# Categorize by life expectancy
life_analysis['Life_Expectancy_Level'] = pd.cut(life_analysis['Life_Expectancy'],
                                                bins=[0, 65, 72, 77, 90],
                                                labels=['Low (≤65)', 'Medium (65-72)', 'High (72-77)', 'Very High (77+)'])

print("\n" + "="*130)
print("❤️  LIFE EXPECTANCY vs WEALTH CONCENTRATION - COMPREHENSIVE ANALYSIS")
print("="*130)
print(f"\n📊 Dataset Overview:")
print(f"  • Countries analyzed: {len(life_analysis)}")
print(f"  • Total billionaires: {life_analysis['Billionaire_Count'].sum():.0f}")
print(f"  • Total wealth tracked: ${life_analysis['Total_Wealth'].sum():,.0f}B")

print(f"\n💚 Life Expectancy Statistics (Years):")
print(f"  • Minimum: {life_analysis['Life_Expectancy'].min():.1f} years")
print(f"  • Maximum: {life_analysis['Life_Expectancy'].max():.1f} years")
print(f"  • Mean: {life_analysis['Life_Expectancy'].mean():.1f} years")
print(f"  • Median: {life_analysis['Life_Expectancy'].median():.1f} years")
print(f"  • Std Dev: {life_analysis['Life_Expectancy'].std():.1f} years")

print(f"\n🔗 CORRELATION ANALYSIS - Does Health Drive Billionaire Concentration?")
print(f"  • Life Expectancy vs Billionaire COUNT: {corr_life_count:+.3f}")
print(f"  • Life Expectancy vs Total WEALTH: {corr_life_wealth:+.3f}")
print(f"  • Life Expectancy vs Average Wealth/Bill: {corr_life_avg:+.3f}")
print(f"  • R-squared (Count): {r_value_life**2:.3f}")
print(f"  • P-value: {p_value_life:.6f}")
if p_value_life < 0.05:
    print(f"  ✅ SIGNIFICANT correlation (p < 0.05)")
else:
    print(f"  ⚠️  Not statistically significant (p >= 0.05)")

print(f"\n💡 STORY ANALYSIS: Are billionaires concentrated in healthier countries?")
if corr_life_count > 0.5:
    print(f"  ✅ YES! STRONG positive correlation: Healthier countries have MORE billionaires")
    print(f"     → Health & development go hand-in-hand with wealth creation")
elif corr_life_count > 0.3:
    print(f"  📈 MODERATELY YES: Some positive relationship exists")
    print(f"     → Health appears to support billionaire production")
elif corr_life_count > 0:
    print(f"  📊 WEAKLY YES: Minimal but positive relationship")
    print(f"     → Health may be one factor among many")
else:
    print(f"  ❌ NO: NO meaningful correlation or negative relationship")
    print(f"     → Health status doesn't predict billionaire concentration")

print(f"\n" + "-"*130)
print("📊 BILLIONAIRE DISTRIBUTION BY LIFE EXPECTANCY LEVEL:")
print("-"*130)
life_dist = life_analysis.groupby('Life_Expectancy_Level', observed=True).agg({
    'Country': 'count',
    'Billionaire_Count': 'sum',
    'Total_Wealth': 'sum',
    'Life_Expectancy': 'mean',
    'Avg_Wealth_per_Billionaire': 'mean'
}).round(1)
life_dist.columns = ['Countries', 'Billionaires', 'Total Wealth (B)', 'Avg Life Exp', 'Avg Wealth/Bill']
print(life_dist)

print(f"\n" + "-"*130)
print("🏆 TOP 15 COUNTRIES BY LIFE EXPECTANCY:")
print("-"*130)
top_life = life_analysis.nlargest(15, 'Life_Expectancy')[['Country', 'Life_Expectancy', 'Billionaire_Count', 'Total_Wealth', 'Avg_Wealth_per_Billionaire']]
for idx, row in top_life.iterrows():
    print(f"{row['Country']:20} | Life Exp: {row['Life_Expectancy']:5.1f} yrs | Billionaires: {row['Billionaire_Count']:3.0f} | Avg Wealth: ${row['Avg_Wealth_per_Billionaire']:8.1f}B")

print(f"\n" + "-"*130)
print("🎯 OUTLIERS - Health/Wealth Mismatch:")
print("-"*130)

# High life expectancy but few billionaires
high_life_low_bill = life_analysis[(life_analysis['Life_Expectancy'] >= 80) & (life_analysis['Billionaire_Count'] < 5)]
if len(high_life_low_bill) > 0:
    print("Countries with HIGH life expectancy but FEW billionaires:")
    for idx, row in high_life_low_bill.nlargest(5, 'Life_Expectancy').iterrows():
        print(f"  • {row['Country']:20} | Life Exp: {row['Life_Expectancy']:.1f} | Billionaires: {row['Billionaire_Count']:.0f}")

print("")

# Low life expectancy but many billionaires
low_life_high_bill = life_analysis[(life_analysis['Life_Expectancy'] < 70) & (life_analysis['Billionaire_Count'] > 5)]
if len(low_life_high_bill) > 0:
    print("Countries with LOW life expectancy but MANY billionaires:")
    for idx, row in low_life_high_bill.nlargest(5, 'Billionaire_Count').iterrows():
        print(f"  • {row['Country']:20} | Life Exp: {row['Life_Expectancy']:.1f} | Billionaires: {row['Billionaire_Count']:.0f}")

print(f"\n" + "="*130)


### Visualization 1: Correlation Plot - Life Expectancy vs Billionaire Count (Scatter with Regression)

In [15]:
# Visualization 1: Scatter Plot with Regression - Life Expectancy vs Billionaire Count
x_line_life = np.array([life_analysis['Life_Expectancy'].min(), life_analysis['Life_Expectancy'].max()])
y_line_life = slope_life * x_line_life + intercept_life

fig_life1 = go.Figure()

# Color by life expectancy level
colors_map = {'Low (≤65)': '#FF6B6B', 'Medium (65-72)': '#FFA500', 'High (72-77)': '#56D8A0', 'Very High (77+)': '#1B5E4A'}
life_analysis['Color'] = life_analysis['Life_Expectancy_Level'].map(colors_map)

fig_life1.add_trace(go.Scatter(
    x=life_analysis['Life_Expectancy'],
    y=life_analysis['Billionaire_Count'],
    mode='markers+text',
    text=life_analysis['Country'],
    textposition='top center',
    textfont=dict(size=7, color='#1B5E4A'),
    marker=dict(
        size=life_analysis['Billionaire_Count'] * 0.6,
        color=life_analysis['Life_Expectancy'],
        colorscale=[
            [0, '#FF6B6B'],        # Red for low life expectancy
            [0.33, '#FFA500'],     # Orange
            [0.66, '#56D8A0'],     # Emerald
            [1, '#1B5E4A']         # Dark emerald for high
        ],
        showscale=True,
        colorbar=dict(
            title="Life<br>Expectancy<br>(Years)",
            thickness=15,
            len=0.7
        ),
        line=dict(color='white', width=1),
        opacity=0.7
    ),
    hovertemplate='<b>%{text}</b><br>' +
                  'Life Expectancy: %{x:.1f} years<br>' +
                  'Billionaires: %{y:.0f}<br>' +
                  '<extra></extra>',
    name='Countries'
))

# Add regression line
fig_life1.add_trace(go.Scatter(
    x=x_line_life,
    y=y_line_life,
    mode='lines',
    name=f'Regression (R²={r_value_life**2:.3f})',
    line=dict(color='#1B5E4A', width=3, dash='dash'),
    hovertemplate='Fit: Life Exp %{x:.1f}yr → Billionaires %{y:.1f}<extra></extra>'
))

fig_life1.update_layout(
    title_text='<b>Life Expectancy vs Billionaire Concentration</b><br>' +
               f'<sub>Correlation: {corr_life_count:.3f} | Are Billionaires in Healthier Countries?</sub>',
    xaxis_title='Life Expectancy (Years)',
    yaxis_title='Number of Billionaires',
    height=700,
    width=1200,
    template='plotly_white',
    font=dict(size=11, color='#1B5E4A'),
    paper_bgcolor='#F0F8F5',
    plot_bgcolor='#FFFFFF',
    hovermode='closest',
    showlegend=True,
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)', bordercolor='#1B5E4A', borderwidth=1)
)

fig_life1.show()

print("\n📊 REGRESSION ANALYSIS - Life Expectancy Effect:")
print(f"  • Slope: {slope_life:.4f} (For each additional year of life expectancy, expect {slope_life:.2f} more billionaires)")
print(f"  • Intercept: {intercept_life:.2f}")
print(f"  • R-squared: {r_value_life**2:.3f} ({r_value_life**2*100:.1f}% of billionaire variance explained by life expectancy)")
print(f"  • Direction: {'POSITIVE = Healthier → More Billionaires' if slope_life > 0 else 'NEGATIVE = Healthier → Fewer Billionaires'}")


NameError: name 'life_analysis' is not defined